In [1]:
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMergeTime\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMerge\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @C:\Users\miao\Documents\Database
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc3\userspace\miao\cluster"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc3\userspace\miao\cluster"


In [4]:
GetDefaultQueue()

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,"[ [OMP_PROC_BIND, spread] ]"
DeploymentBaseDirectory,\\dc3\userspace\miao\cluster
DeployRuntime,True
Name,MSHPC-AllNodes
DotnetRuntime,dotnet
Username,FDY\miao
NumOfAdditionalServiceCores,0
NumOfAdditionalServiceCoresMPISerial,0
NumOfServiceCoresPerMPIprocess,0
ServerName,DC2


In [5]:
static var myBatch = ExecutionQueues[2];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Droplet-EE-LikeEL");

In [6]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE-LikeEL");

Project name is set to 'Droplet-EE-LikeEL'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\Droplet-EE-LikeEL'.


In [7]:
wmg.DefaultDatabase

{ Session Count = 14; Grid Count = 245; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL }

## Create grid

In [8]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xSize = 2;
        double yTop = 1.5 - 20.0 / 176.7;
        double yBottom = -20.0 / 176.7;
        //int kelem = 16;
        int kelem = 8;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 8 * 3 + 1);
                var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                //grd.EdgeTagNames.Add(3, "wall_left");
                //grd.EdgeTagNames.Add(4, "wall_right");
                grd.EdgeTagNames.Add(3, "FreeSlip_left");
                grd.EdgeTagNames.Add(4, "FreeSlip_right");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                        et = 3;
                    if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                        et = 4;

                    return et;
                });

                return grd;
     }
 
 }

In [9]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double pJump(double[] X) {");
           stw.WriteLine("    return 0.046 / 0.4;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return (((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2()).Sqrt() - 0.4);");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] - 0);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_pJump(){
        return new Formula("BoundaryValues.pJump", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [10]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;
            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            int D = 2;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            //C.DbPath = @"\\dc1\userspace\miao\cluster\Droplet-EE-LikeEL-DT";
            C.ProjectName = "Droplet";
            C.SessionName = "Droplet-LikeEL-AMR5-Print-DT";
            C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            // Physical Parameters
            // ===================
            #region physics

            double scale = 4.4175e-4; // For a droplet with radius r = 176.7 micrometre
            C.PhysicalParameters.rho_A = 10 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false; //??

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale, // Real Viscosity
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            #endregion

            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_pJump());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_Zero());
            C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
            //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());



            // boundary conditions
            // ===================
            #region BC


            C.AddBoundaryValue("wall_lower");
            C.AddBoundaryValue("pressure_outlet_upper");
            //C.AddBoundaryValue("wall_left");
            //C.AddBoundaryValue("wall_right");
            C.AddBoundaryValue("FreeSlip_left");
            C.AddBoundaryValue("FreeSlip_right");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 160;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = true;
            C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
            C.AMR_startUpSweeps = 5;
            //C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = 8, levelSets = new[] { 0, 1 }, FieldWidth = 2 });
            //C.AMR_startUpSweeps = 8;

            #endregion


            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 5e-7;
            double dt = 2e-1;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 10000;
            C.saveperiod = 1;

            #endregion

            return C;
        }

## Send and run jobs

In [11]:
    var C_CTRLFILE = Aland(2, 0);//Create control file.

Grid Edge Tags changed.


In [12]:
    var JobLocal = C_CTRLFILE.CreateJob();

In [13]:
JobLocal.Status

PreActivation

In [14]:
//SetDefaultQueue("MSHPC-AllNodes")

In [15]:
    JobLocal.NumberOfMPIProcs = 4;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate();
    JobLocal.ShowOutput();
    //BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Droplet-LikeEL-AMR5-Print-DT ... 
Opening existing database 'C:\Users\miao\Documents\Database\Droplet-EE-LikeEL'.
Set Database: { Session Count = 14; Grid Count = 239; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL }
Grid is not in database yet...
Grid successfully saved: e00b51b1-6705-419d-942f-b87b69c5162f


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-ZwoLevelSetSolver2024Feb12_145815.013730
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [16]:
JobLocal.ShowOutput();

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [ ]:
//JobLocal.Stdout

In [21]:
wmg.Sessions

#0: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/9/2024 6:52:46 AM	cff62878...
#1: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 10:03:40 PM	6f3769fc...
#2: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 9:37:05 PM	12e06e8a...
#3: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print	1/24/2024 2:44:32 PM	3fcac1ec...
#4: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 5:03:24 PM	f8e280ed...
#5: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 10:00:23 AM	c93c6dbe...
#6: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart*	11/26/2023 7:12:40 PM	3a7cc021...
#7: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart*	11/25/2023 10:11:11 AM	7eb52e19...
#8: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	11/25/2023 12:29:03 AM	a302498c...
#9: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart*	11/24/2023 11:40:40 AM	a6f729f7...
#10: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5*	11/22/2023 7:33:01 PM	88cf424b...


In [10]:
wmg.Sessions[0].Timesteps.Count

113

In [ ]:
//var outPath = wmg.Sessions[2].Timesteps.Every(1).Skip(6 + 935).Export().WithSupersampling(3).Do();

## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [18]:
databases[0].Sessions

#0: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/9/2024 6:52:46 AM	cff62878...
#1: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 10:03:40 PM	6f3769fc...
#2: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 9:37:05 PM	12e06e8a...
#3: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print	1/24/2024 2:44:32 PM	3fcac1ec...
#4: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 5:03:24 PM	f8e280ed...
#5: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 10:00:23 AM	c93c6dbe...
#6: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart*	11/26/2023 7:12:40 PM	3a7cc021...
#7: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart*	11/25/2023 10:11:11 AM	7eb52e19...
#8: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	11/25/2023 12:29:03 AM	a302498c...
#9: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart*	11/24/2023 11:40:40 AM	a6f729f7...
#10: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5*	11/22/2023 7:33:01 PM	88cf424b...


In [11]:
var session = databases[0].Sessions[0];

In [12]:
session.Timesteps.Count

628

In [13]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("VelocityY").ProbeAt(0.405, 0),
        t => "VelocityY"
        );

In [14]:
mydataset.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 2 
 
 
 
 
 4 
 
 
 
 
 6 
 
 
 
 
 8 
 
 
 
 
 10 
 
 
 
 
 12 
 
 
 
 
 14 
 
 
 
 
 0 
 
 
 
 
 0.0005 
 
 
 
 
 0.001 
 
 
 
 
 0.0015 
 
 
 
 
 0.002 
 
 
 
 
 0.0025 
 
 
 
 
 0.003 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 VelocityY 
 
 
 VelocityY 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M577.4,57.1 L630.8,57.1 M45.6,564.0 L46.7,136.8 L47.9,100.9 L49.0,105.7 L50.1,108.5 L51.2,106.2
 L52.4,108.0 L53.5,109.9 L54.6,111.7 L55.8,113.5 L56.9,115.3 L58.0,117.1 L59.1,118.9 L60.3,120.7
 L61.4,122.5 L62.5,124.2 L63.7,126.0 L64.8,126.5 L65.9,128.4 L67.1,130.3 L68.2,132.3 L69.3,134.2
 L70.4,136.1 L71.6,138.0 L72.7,139.9 L73.8,141.8 L75.0,140.6 L76.1,142.4 L77.2,144.2 L78.3,146.1
 L79.5,147.9 L80.6,149.7 L81.7,151.5 L82.9,153.3 L84.0,155.1 L85.1,156.9 L86.2,158.6 L87.4,160.4
 L88.5,162.1 L89.6,163.9 L90.8,165.6 L91.9,167.3 L93.0,169.0 L94.2,170.7 L95.3,167.1 L96.4,168.8
 L97.5,170.4 L98.7,172.0 L99.8,173.6 L100.9,175.2 L102.1,176.9 L103.2,178.5 L104.3,178.8 L105.4,180.3
 L106.6,181.9 L107.7,183.4 L108.8,185.0 L110.0,186.5 L111.1,188.1 L112.2,189.6 L113.3,191.1 L114.5,192.6
 L115.6,194.1 L116.7,195.6 L117.9,197.1 L119.0,198.6 L120.1,200.1 L121.3,201.5 L122.4,203.0 L123.5,204.5
 L124.6,205.9 L125.8,207.4 L126.9,208.8 L128.0,213.7 L129.2,215.1 L130.3,216.5 L131.4,217.8 L132.5,219.2
 L133.7,220.6 L134.8,221.9 L135.9,223.3 L137.1,224.6 L138.2,226.0 L139.3,227.3 L140.4,228.6 L141.6,230.0
 L142.7,231.1 L143.8,232.4 L145.0,233.8 L146.1,235.1 L147.2,236.3 L148.4,237.6 L149.5,238.9 L150.6,240.2
 L151.7,241.5 L152.9,242.7 L154.0,244.0 L155.1,245.3 L156.3,246.5 L157.4,247.8 L158.5,249.0 L159.6,250.2
 L160.8,251.5 L161.9,252.7 L163.0,253.9 L164.2,255.2 L165.3,256.4 L166.4,257.6 L167.5,258.8 L168.7,260.0
 L169.8,261.2 L170.9,262.4 L172.1,263.6 L173.2,264.8 L174.3,265.9 L175.5,267.1 L176.6,268.3 L177.7,269.4
 L178.8,270.6 L180.0,271.8 L181.1,272.9 L182.2,274.1 L183.4,275.2 L184.5,276.4 L185.6,277.5 L186.7,278.6
 L187.9,279.8 L189.0,280.9 L190.1,282.0 L191.3,283.1 L192.4,284.3 L193.5,285.4 L194.6,286.5 L195.8,287.6
 L196.9,288.7 L198.0,289.8 L199.2,290.9 L200.3,292.0 L201.4,293.1 L202.6,294.2 L203.7,295.2 L204.8,296.3
 L205.9,297.4 L207.1,298.5 L208.2,299.5 L209.3,300.6 L210.5,301.7 L211.6,302.7 L212.7,303.8 L213.8,304.8
 L215.0,305.9 L216.1,306.9 L217.2,308.0 L218.4,309.0 L219.5,310.1 L220.6,311.1 L221.7,312.1 L222.9,313.2
 L224.0,314.2 L225.1,315.2 L226.3,316.2 L227.4,317.2 L228.5,318.3 L229.6,319.2 L230.8,320.2 L231.9,321.2
 L233.0,322.2 L234.2,323.2 L235.3,325.3 L236.4,326.1 L237.6,326.9 L238.7,327.8 L239.8,328.7 L240.9,329.6
 L242.1,330.6 L243.2,331.5 L244.3,332.5 L245.5,333.4 L246.6,331.3 L247.7,332.2 L248.8,333.1 L250.0,329.8
 L251.1,330.7 L252.2,331.6 L253.4,332.5 L254.5,333.4 L255.6,334.4 L256.7,335.3 L257.9,336.2 L259.0,337.1
 L260.1,338.0 L261.3,338.9 L262.4,339.7 L263.5,340.6 L264.7,341.5 L265.8,342.4 L266.9,343.3 L268.0,344.1
 L269.2,345.0 L270.3,345.9 L271.4,346.7 L272.6,347.6 L273.7,348.4 L274.8,349.3 L275.9,350.1 L277.1,351.0
 L278.2,351.8 L279.3,352.7 L280.5,353.5 L281.6,354.3 L282.7,355.1 L283.8,356.0 L285.0,356.8 L286.1,357.6
 L287.2,358.4 L288.4,359.2 L289.5,360.0 L290.6,360.8 L291.8,361.6 L292.9,362.4 L294.0,363.2 L295.1,364.0
 L296.3,364.8 L297.4,365.6 L298.5,366.4 L299.7,367.2 L300.8,367.9 L301.9,368.7 L303.0,369.5 L304.2,370.3
 L305.3,371.0 L306.4,371.8 L307.6,372.5 L308.7,373.3 L309.8,374.1 L310.9,374.8 L312.1,375.5 L313.2,376.3
 L314.3,375.5 L315.5,376.6 L316.6,377.5 L317.7,378.3 L318.9,379.2 L320.0,380.0 L321.1,380.7 L322.2,381.5
 L323.4,382.2 L324.5,383.0 L325.6,383.7 L326.8,384.4 L327.9,390.9 L329.0,391.9 L330.1,39

In [15]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(518).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL__Droplet-LikeEL-AMR5-Print-DT__d360fae5-6e61-46f1-97af-dcef956f8d36


In [16]:
var DispY = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementY").Single();

In [18]:
DispY

DisplacementY

In [ ]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("Phi2"),
        t => "DisplacementY"
        );

In [ ]:
mydataset.PlotNow()

## Restart for steady calculation

In [11]:
databases[0].Sessions

#0: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print-Steady*	2/12/2024 4:56:03 PM	08b3b17d...
#1: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print-DT_Restart*	2/12/2024 10:38:00 AM	5111ab71...
#2: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print-DT*	2/10/2024 3:51:27 PM	d360fae5...
#3: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/9/2024 6:52:46 AM	cff62878...
#4: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 10:03:40 PM	6f3769fc...
#5: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 9:37:05 PM	12e06e8a...
#6: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print	1/24/2024 2:44:32 PM	3fcac1ec...
#7: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 5:03:24 PM	f8e280ed...
#8: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 10:00:23 AM	c93c6dbe...
#9: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart*	11/26/2023 7:12:40 PM	3a7cc021...
#10: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart*	11/25/2023 10:11:11 AM	7eb5

In [12]:
//databases[0].Sessions[0].Delete(true)

In [13]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[1];
Sloc

Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print-DT_Restart*	2/12/2024 10:38:00 AM	5111ab71...

In [14]:
var c3 = Sloc.CreateRestartControl() as ZLS_Control;

In [15]:
c3.GetType()

ZwoLevelSetSolver.ZLS_Control

In [16]:
c3.GridGuid

59e97799-e36b-4af1-a4ab-f15c7074996b

In [17]:
Sloc.Timesteps.Last().GridID

59e97799-e36b-4af1-a4ab-f15c7074996b

In [18]:
c3.GridGuid = Sloc.Timesteps.Last().GridID;

In [19]:
c3.GridGuid

59e97799-e36b-4af1-a4ab-f15c7074996b

In [20]:
            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Steady;

            c3.ProjectName = "Droplet";
            c3.SessionName = "Droplet-LikeEL-AMR5-Print-Steady";
            c3.ProjectDescription = "Droplet running on pc";

            c3.AdaptiveMeshRefinement = false;
            //c3.DynamicLoadBalancing_RedistributeAtStartup = false;
            

            c3.TimesteppingMode = compMode;

In [21]:
c3.GridPartType = GridPartType.OtherSession;

In [ ]:
c3.GridPartType = 

In [17]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [23]:
var JobLocal2 = c3.CreateJob();
JobLocal2.Status

PreActivation

In [24]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

 ------------ MSHPC FailedOrCanceled; original Failed
Deployments so far (2): (Job token: 896566, FinishedSuccessful 'Droplet-EE-LikeEL-ZwoLevelSetSolver2024Feb12_135748.711354' @ MS HPC client  MSHPC-AllNodes @DC2, @\\dc3\userspace\miao\cluster, FinishedSuccessful), (Job token: 896567, FailedOrCanceled 'Droplet-EE-LikeEL-ZwoLevelSetSolver2024Feb12_140330.645107' @ MS HPC client  MSHPC-AllNodes @DC2, @\\dc3\userspace\miao\cluster, FailedOrCanceled);
Success: 1
Note: found 1 successful deployment(s), but job is configured to require a successful result session ('this.SessionReqForSuccess' is true), and none is found. 0 sessions correlated to this job fount in total.
job submit count: 2


unable to determine job status - unknown


Deploying job Droplet-LikeEL-AMR5-Print-Steady ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-ZwoLevelSetSolver2024Feb12_165233.082214
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [22]:
databases[0].Sessions

#0: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print-Steady	2/12/2024 4:56:03 PM	08b3b17d...
#1: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print-DT_Restart*	2/12/2024 10:38:00 AM	5111ab71...
#2: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print-DT*	2/10/2024 3:51:27 PM	d360fae5...
#3: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/9/2024 6:52:46 AM	cff62878...
#4: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 10:03:40 PM	6f3769fc...
#5: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print*	2/8/2024 9:37:05 PM	12e06e8a...
#6: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5-Print	1/24/2024 2:44:32 PM	3fcac1ec...
#7: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 5:03:24 PM	f8e280ed...
#8: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart_Restart*	11/27/2023 10:00:23 AM	c93c6dbe...
#9: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart_Restart*	11/26/2023 7:12:40 PM	3a7cc021...
#10: Droplet-EE-LikeEL	Droplet-LikeEL-AMR5_Restart_Restart*	11/25/2023 10:11:11 AM	7eb52

In [23]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL__Droplet-LikeEL-AMR5-Print-Steady__08b3b17d-1089-4c4d-bce9-83b1a8aec90b


In [ ]:
databases[0].Sessions[0].